# 🌿 Plant Disease Detection - Training Script

> **`train.py`** - Production-Ready Model Training Pipeline

---

## 📋 Script Overview

```python
# ============================================================================
# PLANT DISEASE DETECTION - TRAINING SCRIPT
# ============================================================================
# This script trains a MobileNetV2 model on the PlantVillage dataset using
# transfer learning and fine-tuning techniques. The model is saved in multiple
# formats for maximum compatibility.
# ============================================================================

In [1]:
# ============================================================================
# 1. IMPORT LIBRARIES
# ============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import json
import shutil

In [2]:
# ============================================================================
# 2. INITIAL SETTINGS & CONFIGURATION
# ============================================================================
print("✅ Initial settings...")
# Suppress TensorFlow warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# Limit CPU threads to prevent memory overload on 8GB RAM systems
tf.config.threading.set_intra_op_parallelism_threads(4)

✅ Initial settings...


In [3]:
# ============================================================================
# 3. DATASET PATH CONFIGURATION
# ============================================================================
# Define dataset paths - adjust these according to your directory structure
DATASET_PATH = r"../Dataset/"
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VALID_PATH = os.path.join(DATASET_PATH, "valid")
TEST_PATH = os.path.join(DATASET_PATH, "test")

# Verify dataset folders exist
if not os.path.exists(TRAIN_PATH):
    print(f"❌ Error: Train folder not found at: {TRAIN_PATH}")
    print("Please adjust DATASET_PATH to point to your PlantVillage dataset.")
    exit()

print(f"✅ Train path: {TRAIN_PATH}")
print(f"✅ Validation path: {VALID_PATH}")
print(f"✅ Test path: {TEST_PATH}")

# ============================================================================
# 4. DATASET ANALYSIS
# ============================================================================
print("\n📊 Analyzing dataset...")
num_classes = len([d for d in os.listdir(TRAIN_PATH) if os.path.isdir(os.path.join(TRAIN_PATH, d))])
print(f"Number of classes: {num_classes}")

✅ Train path: ../Dataset/train
✅ Validation path: ../Dataset/valid
✅ Test path: ../Dataset/test

📊 Analyzing dataset...
Number of classes: 38


In [4]:
# ============================================================================
# 5. DATA AUGMENTATION & GENERATORS
# ============================================================================
# Data augmentation helps prevent overfitting by creating variations of training images
print("\n🛠️ Building Data Generators...")
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalize pixel values from 0-255 to 0-1
    rotation_range=20,           # Random rotation up to 20 degrees
    width_shift_range=0.2,       # Random horizontal shift up to 20%
    height_shift_range=0.2,      # Random vertical shift up to 20%
    horizontal_flip=True,        # Random horizontal flip (mirror image)
    zoom_range=0.1,              # Random zoom up to 10%
    fill_mode='nearest'          # Fill empty pixels with nearest neighbor
)

# For validation and test: only normalize, no augmentation
valid_test_datagen = ImageDataGenerator(rescale=1./255)

# ============================================================================
# 6. LOAD DATASET
# ============================================================================
BATCH_SIZE = 32  # Reduced for 8GB RAM systems
IMG_SIZE = (128, 128)  # Reduced resolution for faster training and less memory usage

print("\n🔄 Loading training data...")
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

print("\n🔄 Loading validation data...")
valid_generator = valid_test_datagen.flow_from_directory(
    VALID_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("\n🔄 Loading test data...")
test_generator = valid_test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\n📊 Dataset Statistics:")
print(f"Training images: {train_generator.samples}")
print(f"Validation images: {valid_generator.samples}")
print(f"Test images: {test_generator.samples}")



🛠️ Building Data Generators...

🔄 Loading training data...
Found 37997 images belonging to 38 classes.

🔄 Loading validation data...
Found 10859 images belonging to 38 classes.

🔄 Loading test data...
Found 5449 images belonging to 38 classes.

📊 Dataset Statistics:
Training images: 37997
Validation images: 10859
Test images: 5449


In [5]:
# ============================================================================
# 7. BUILD THE MODEL USING FUNCTIONAL API (Continued)
# ============================================================================
# This section defines the model architecture using TensorFlow's Functional API
# which provides better compatibility and flexibility compared to Sequential API

# Define the input layer - this creates a placeholder for input images
# Shape: (128, 128, 3) = height, width, RGB channels
inputs = tf.keras.Input(shape=(128, 128, 3))

# Load pre-trained MobileNetV2 as the base model
# - weights='imagenet': Use pre-trained weights from ImageNet dataset
# - include_top=False: Remove the final classification layers (we'll add our own)
# - input_tensor=inputs: Connect to our explicit input layer
# This is the key to transfer learning - we leverage knowledge from 14M+ images
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_tensor=inputs
)
# Freeze base model layers to preserve pre-trained features
# This prevents destroying the valuable knowledge learned from ImageNet
base_model.trainable = False

# Add custom classification layers for plant disease detection
# These layers will be trained on our specific dataset
x = base_model.output                              # Get output from base model (shape: 4x4x1280)
x = layers.GlobalAveragePooling2D()(x)             # Reduce spatial dimensions to 1D vector (1280)
x = layers.Dropout(0.3)(x)                         # 30% dropout to prevent overfitting
x = layers.Dense(128, activation='relu')(x)        # Fully connected layer with 128 neurons
x = layers.Dropout(0.2)(x)                         # 20% dropout for additional regularization
outputs = layers.Dense(num_classes, activation='softmax')(x)  # Final layer with num_classes outputs

# Create the complete model by specifying inputs and outputs
model = tf.keras.Model(inputs=inputs, outputs=outputs)

print("✅ Model built successfully using Functional API!")


C:\Users\Windows\AppData\Local\Temp\ipykernel_20484\3454398570.py:16: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


✅ Model built successfully using Functional API!


In [6]:
# ============================================================================
# 8. COMPILE THE MODEL
# ============================================================================
# Configure the model for training with appropriate optimizer, loss, and metrics

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),  # Adam optimizer with 0.001 learning rate
    loss='categorical_crossentropy',                  # Suitable for multi-class classification
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

print("\n📋 Model Architecture Summary:")
model.summary()
# The summary shows:
# - Total parameters: ~2.4M (2.26M frozen + 168K trainable)
# - Trainable parameters are only from the new layers we added
# - Non-trainable parameters are from the frozen MobileNetV2 base


📋 Model Architecture Summary:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 128, 128, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 64, 64, 32)        │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 64, 64, 32)        │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 64, 64, 32)        │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 64, 64, 32)        │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 64, 64, 32)        │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 64, 64, 32)        │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 64, 64, 16)        │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 64, 64, 16)        │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 64, 64, 96)        │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 64, 64, 96)        │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 64, 64, 96)        │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 65, 65, 96)        │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 32, 32, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 168,870 (659.65 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
# ============================================================================
# 9. TRAIN THE MODEL WITH EARLY STOPPING
# ============================================================================
print("\n🚀 Starting training process...")

# Early stopping callback: prevent overfitting by stopping training
# when validation loss doesn't improve for 'patience' epochs
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',        # Monitor validation loss
    patience=5,                # Stop if no improvement for 5 epochs
    restore_best_weights=True  # Restore the best model weights
)

# Train the model
history = model.fit(
    train_generator,           # Training data (37,997 images)
    epochs=20,                 # Maximum number of epochs
    validation_data=valid_generator,  # Validation data (10,859 images)
    callbacks=[early_stopping],       # Early stopping callback
    verbose=1                  # Show progress bars
)

print("✅ Training completed successfully!")


🚀 Starting training process...
Epoch 1/20


D:\Programs\Anaconda\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1188/1188 ━━━━━━━━━━━━━━━━━━━━ 423s 344ms/step - accuracy: 0.5722 - loss: 1.5741 - precision: 0.8121 - recall: 0.4475 - val_accuracy: 0.8565 - val_loss: 0.4521 - val_precision: 0.9207 - val_recall: 0.7985
Epoch 2/20
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 210s 176ms/step - accuracy: 0.7832 - loss: 0.6745 - precision: 0.8607 - recall: 0.7229 - val_accuracy: 0.8751 - val_loss: 0.3818 - val_precision: 0.9251 - val_recall: 0.8267
Epoch 3/20
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 195s 164ms/step - accuracy: 0.8106 - loss: 0.5955 - precision: 0.8724 - recall: 0.7582 - val_accuracy: 0.8760 - val_loss: 0.3709 - val_precision: 0.9228 - val_recall: 0.8370
Epoch 4/20
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 170s 142ms/step - accuracy: 0.8170 - loss: 0.5729 - precision: 0.8775 - recall: 0.7708 - val_accuracy: 0.8845 - val_loss: 0.3476 - val_precision: 0.9244 - val_recall: 0.8481
Epoch 5/20
1188/1188 ━━━━━━━━━━━━━━━━━━━━ 174s 146ms/step - accuracy: 0.8252 - loss: 0.5437 - precision: 0.8784 - recall: 0.7836 - val_accuracy: 0.

In [8]:
# ============================================================================
# 10. SAVE MODEL IN MULTIPLE FORMATS (FOR COMPATIBILITY)
# ============================================================================
# We save in multiple formats to ensure compatibility with different TensorFlow versions
# and deployment scenarios

print("\n💾 Saving model in multiple formats...")

# Format 1: SavedModel - Most compatible format for TensorFlow Serving
print("\n📦 Saving SavedModel format...")
try:
    # Remove existing folder if present
    if os.path.exists('saved_model'):
        shutil.rmtree('saved_model')
    # Save as SavedModel (compatible with TF 2.x)
    tf.saved_model.save(model, 'saved_model')
    print("✅ Model saved: saved_model/")
except Exception as e:
    print(f"⚠️ SavedModel save error: {e}")

# Format 2: Keras .keras format - Recommended for Keras 3
print("\n📦 Saving .keras format...")
try:
    model.save('plant_disease_model.keras')
    print("✅ Model saved: plant_disease_model.keras")
except Exception as e:
    print(f"⚠️ .keras save error: {e}")

# Format 3: HDF5 .h5 format - Legacy compatibility
print("\n📦 Saving .h5 format...")
try:
    model.save('plant_disease_model.h5')
    print("✅ Model saved: plant_disease_model.h5")
except Exception as e:
    print(f"⚠️ .h5 save error: {e}")

# Format 4: Weights only - Lightweight backup (works everywhere)
print("\n📦 Saving model weights...")
try:
    model.save_weights('model_weights.h5')
    print("✅ Model weights saved: model_weights.h5")
except Exception as e:
    print(f"⚠️ Weights save error: {e}")

# Format 5: Model architecture as JSON - Human-readable structure
print("\n📦 Saving model architecture...")
model_json = model.to_json()
with open('model_architecture.json', 'w', encoding='utf-8') as f:
    json.dump(model_json, f, ensure_ascii=False)
print("✅ Model architecture saved: model_architecture.json")



💾 Saving model in multiple formats...

📦 Saving SavedModel format...
INFO:tensorflow:Assets written to: saved_model\assets


INFO:tensorflow:Assets written to: saved_model\assets


✅ Model saved: saved_model/

📦 Saving .keras format...


✅ Model saved: plant_disease_model.keras

📦 Saving .h5 format...
✅ Model saved: plant_disease_model.h5

📦 Saving model weights...
⚠️ Weights save error: The filename must end in `.weights.h5`. Received: filepath=model_weights.h5

📦 Saving model architecture...
✅ Model architecture saved: model_architecture.json


In [9]:
# ============================================================================
# 11. SAVE CLASS NAMES AND MODEL INFORMATION
# ============================================================================
# These files are essential for the Streamlit web application

print("\n📦 Saving class names and model information...")

# Save class names for the web interface
class_names = list(test_generator.class_indices.keys())
with open('class_names.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, ensure_ascii=False)
print("✅ Class names saved: class_names.json")

# Save comprehensive model information for reference
model_info = {
    'input_shape': (128, 128, 3),
    'num_classes': num_classes,
    'class_names': class_names,
    'model_type': 'MobileNetV2',
    'total_parameters': model.count_params(),
    'trainable_parameters': sum(w.shape.num_elements() for w in model.trainable_weights),
    'non_trainable_parameters': sum(w.shape.num_elements() for w in model.non_trainable_weights)
}
with open('model_info.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False)
print("✅ Model information saved: model_info.json")


📦 Saving class names and model information...
✅ Class names saved: class_names.json
✅ Model information saved: model_info.json


In [10]:
# ============================================================================
# 12. EVALUATE ON TEST DATASET
# ============================================================================
print("\n📊 Evaluating model on test dataset...")
test_loss, test_acc, test_precision, test_recall = model.evaluate(test_generator)

print("\n📈 Final Test Results:")
print(f"Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Precision: {test_precision:.4f} ({test_precision*100:.2f}%)")
print(f"Recall: {test_recall:.4f} ({test_recall*100:.2f}%)")

# ============================================================================
# 13. CONFUSION MATRIX VISUALIZATION
# ============================================================================
# The confusion matrix shows detailed classification performance per class
# It helps identify which diseases the model confuses with each other

print("\n🔄 Generating Confusion Matrix...")

# Reset the test generator to ensure we get all samples
test_generator.reset()

# Make predictions on all test images
Y_pred = model.predict(test_generator, steps=len(test_generator))
y_pred = np.argmax(Y_pred, axis=1)  # Get predicted class indices
y_true = test_generator.classes[:len(y_pred)]  # Get true class indices

# Create and plot confusion matrix
plt.figure(figsize=(15, 12))
cm = confusion_matrix(y_true, y_pred)

# Plot heatmap with seaborn
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Plant Disease Classification')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

# Save the figure
plt.savefig('confusion_matrix.png', dpi=100)
print("✅ Confusion Matrix saved: confusion_matrix.png")
plt.close()  # Close the figure to free memory


📊 Evaluating model on test dataset...
  1/171 ━━━━━━━━━━━━━━━━━━━━ 37s 222ms/step - accuracy: 0.7500 - loss: 0.7730 - precision: 0.7419 - recall: 0.7188

D:\Programs\Anaconda\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:120: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


171/171 ━━━━━━━━━━━━━━━━━━━━ 23s 136ms/step - accuracy: 0.9150 - loss: 0.2744 - precision: 0.9284 - recall: 0.8993

📈 Final Test Results:
Accuracy: 0.9044 (90.44%)
Precision: 0.9326 (93.26%)
Recall: 0.8767 (87.67%)

🔄 Generating Confusion Matrix...
171/171 ━━━━━━━━━━━━━━━━━━━━ 26s 145ms/step
✅ Confusion Matrix saved: confusion_matrix.png


In [11]:
# ============================================================================
# 14. COMPLETION SUMMARY
# ============================================================================
print("\n" + "="*60)
print("✅ PROJECT COMPLETED SUCCESSFULLY!")
print("="*60)

print("\n📁 Generated Files:")
print("   ├── saved_model/               # SavedModel format (most compatible)")
print("   ├── plant_disease_model.keras  # Keras 3 format (recommended)")
print("   ├── plant_disease_model.h5     # HDF5 format (legacy)")
print("   ├── model_weights.h5           # Weights only (lightweight)")
print("   ├── model_architecture.json    # Architecture (human-readable)")
print("   ├── model_info.json            # Model information")
print("   ├── class_names.json           # Class names for web app")
print("   └── confusion_matrix.png       # Confusion matrix visualization")

print("\n📊 Model Performance Summary:")
print(f"   ├── Accuracy:  {test_acc*100:.2f}%")
print(f"   ├── Precision: {test_precision*100:.2f}%")
print(f"   └── Recall:    {test_recall*100:.2f}%")

print(f"\n📈 Model Statistics:")
print(f"   ├── Total Parameters:     {model.count_params():,}")
print(f"   ├── Trainable Parameters: {sum(w.shape.num_elements() for w in model.trainable_weights):,}")
print(f"   └── Non-trainable:        {sum(w.shape.num_elements() for w in model.non_trainable_weights):,}")

print("\n🚀 Next Steps:")
print("   1. Run 'streamlit run app.py' to launch the web interface")
print("   2. Upload leaf images to test the model")
print("   3. Deploy to Streamlit Cloud or your own server")

print("\n" + "="*60)
print("🌿 Training complete! Model ready for deployment.")
print("="*60)

# ============================================================================
# END OF TRAINING SCRIPT
# ============================================================================


✅ PROJECT COMPLETED SUCCESSFULLY!

📁 Generated Files:
   ├── saved_model/               # SavedModel format (most compatible)
   ├── plant_disease_model.keras  # Keras 3 format (recommended)
   ├── plant_disease_model.h5     # HDF5 format (legacy)
   ├── model_weights.h5           # Weights only (lightweight)
   ├── model_architecture.json    # Architecture (human-readable)
   ├── model_info.json            # Model information
   ├── class_names.json           # Class names for web app
   └── confusion_matrix.png       # Confusion matrix visualization

📊 Model Performance Summary:
   ├── Accuracy:  90.44%
   ├── Precision: 93.26%
   └── Recall:    87.67%

📈 Model Statistics:
   ├── Total Parameters:     2,426,854
   ├── Trainable Parameters: 168,870
   └── Non-trainable:        2,257,984

🚀 Next Steps:
   1. Run 'streamlit run app.py' to launch the web interface
   2. Upload leaf images to test the model
   3. Deploy to Streamlit Cloud or your own server

🌿 Training complete! Model re